# Phase 3 GPU Run

Notebook này dùng cho Kaggle/Colab T4 để chạy PhoBERT gating: 100% train, seed 42, no augmentation. Trước khi chạy, đặt repo `vsfc-low-resource-da` trong working directory của notebook hoặc chỉnh `PROJECT_DIR` ở cell đầu.

In [ ]:
from pathlib import Path
import os
import subprocess

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTHONUNBUFFERED', '1')
os.environ.setdefault('WANDB_DISABLED', 'true')

REPO_URL = 'https://github.com/AI-Peak/vsfc-low-resource-da.git'
candidates = [
    Path('/kaggle/working/vsfc-low-resource-da'),
    Path('/content/vsfc-low-resource-da'),
    Path.cwd(),
]
PROJECT_DIR = next((p for p in candidates if (p / 'src').exists()), candidates[0])
if not (PROJECT_DIR / 'src').exists():
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR, check=True)
os.chdir(PROJECT_DIR)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=False)
print('PROJECT_DIR =', PROJECT_DIR)
print('contains src:', (PROJECT_DIR / 'src').exists())
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES'))

In [ ]:
!python -m pip install -q -r requirements.txt

In [ ]:
import accelerate
import transformers

print('transformers =', transformers.__version__)
print('accelerate =', accelerate.__version__)
assert transformers.__version__.startswith('4.'), transformers.__version__

In [ ]:
!python scripts/check_gpu.py

In [ ]:
!python scripts/setup_vncorenlp.py

In [ ]:
!python scripts/download_data.py

In [ ]:
!python -m src.data.subsample --train-csv data/raw/train.csv --seed 42

In [ ]:
!python -m src.experiments.run_phobert --ratio 0.05 --seed 42 --augmentation none --results-dir results/smoke --max-train-samples 128 --max-dev-samples 64 --max-test-samples 64 --num-epochs 1 --logging-steps 1 --disable-gating --force-preprocess --overwrite

In [ ]:
!python -m src.experiments.run_phobert --ratio 1.00 --seed 42 --augmentation none --logging-steps 25 --overwrite

In [ ]:
import json
from pathlib import Path

metrics_path = Path('results/logs/phobert_none_1.00_42.json')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(json.dumps(metrics['test'], ensure_ascii=False, indent=2))